[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/00_%EC%8B%9C%EC%9E%91%ED%95%98%EA%B8%B0_Colab%EA%B3%BC_%ED%95%B5%EC%8B%AC%ED%8C%A8%ED%84%B4.ipynb)

# 00 · 시작하기 — Colab 사용법과 분석의 5가지 핵심 패턴

**⏱ 30분** · 이 과정 16시간 동안 쓰는 도구와 코드 패턴을 먼저 익힙니다.

## 이 과정에서 하는 일

설비가 고장 나기 전에 **상태를 보고 정비하는 것**을 상태기반정비(CBM, Condition-Based Maintenance)라고 합니다.
CBM은 아래 흐름으로 움직입니다. 이 과정에서는 세 가지 실제 데이터로 이 흐름을 한 단계씩 직접 계산해 봅니다.

```
센서·기록 데이터 → 특징(건전성 지표) → ① 이상탐지 → ② 고장진단 → ③ 잔여수명(RUL) 예측 → 정비 결정
```

| 모듈 | 데이터 | CBM에서 배우는 것 |
|---|---|---|
| 1 | Backblaze 하드디스크 6만 대의 90일 기록 | 고장은 얼마나 드문가, 상태 신호로 경보를 만들고 **비용**으로 기준 정하기 |
| 2 | 유압 시험장치 2,205 사이클 | 센서 → 특징 → **① 이상탐지**, **② 고장진단** (머신러닝 분류) |
| 3 | 압출기 필터 막힘 실험 41회 | 열화 지표 → **③ 잔여수명(RUL) 예측** → 교체 시점 결정 |

## 1. Colab 사용법 (5분)

- 코드 셀을 클릭하고 **Shift + Enter** 를 누르면 실행됩니다. 셀 왼쪽에 숫자 `[1]`이 생기면 실행된 것입니다.
- **위에서부터 순서대로** 실행하세요. 아래 셀은 위 셀에서 만든 변수를 사용합니다.
- 한동안 쓰지 않거나 창을 닫으면 Colab이 **런타임을 초기화**합니다. 그러면 변수가 모두 사라지므로 **맨 위 준비 셀부터 다시** 실행하세요.
- 내 작업을 남기려면 메뉴 **파일 → Drive에 사본 저장** 을 먼저 누르세요. 사본을 저장하지 않으면 수정한 내용이 사라집니다.

### 셀 표시 읽는 법

| 표시 | 뜻 | 할 일 |
|---|---|---|
| ▶ | 제공 코드 | 그대로 실행하고 **결과만** 읽습니다. 코드를 외울 필요는 없습니다. |
| ✍️ | 직접 입력 | 보여 준 코드를 **손으로 타이핑**해서 실행합니다. 반복해서 나오는 핵심 패턴입니다. |
| 📝 | 과제 | 앞에서 입력한 코드를 **조금 바꿔서** 스스로 풉니다. |
| 💬 | 결과 해설 | 방금 나온 숫자·그래프를 어떻게 읽는지 설명합니다. |
| ✅ | 체크포인트 | 내 결과가 맞는지 확인합니다. |

### 오류가 났을 때

1. 빨간 오류 메시지의 **마지막 줄**을 읽습니다. (`NameError` = 변수 이름이 없음 → 위 셀을 실행했는지, 철자가 같은지 확인)
2. 괄호 `( )`, 따옴표 `' '`, 쉼표 `,` 가 짝이 맞는지 확인합니다.
3. 그래도 안 되면 옆 사람과 비교하거나 손을 드세요.

## 2. 준비 셀

모든 노트북의 첫 코드 셀은 **준비 셀**입니다. 실습 데이터를 GitHub에서 내려받고, 그래프에 한글이 나오도록 글꼴을 설정합니다.
`import pandas as pd` 같은 줄은 "표 계산 도구(pandas)를 `pd`라는 짧은 이름으로 불러온다"는 뜻입니다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/filter_runs.csv",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

✅ `준비 완료!` 가 출력되면 됩니다.

---
## 3. 다섯 가지 핵심 패턴

데이터 분석 코드는 복잡해 보여도 대부분 **다섯 가지 패턴의 조합**입니다. 이 과정 내내 같은 패턴이 반복됩니다.

| 패턴 | 하는 일 | 대표 코드 |
|---|---|---|
| P1 읽고 보기 | 파일을 표로 읽고 모양을 본다 | `pd.read_csv(...)`, `df.head()`, `df.shape` |
| P2 골라내기 | 조건에 맞는 행만 남긴다 | `df[df['열'] == 값]` |
| P3 묶어서 요약 | 그룹별 평균·개수를 구한다 | `df.groupby('그룹열')['값열'].mean()` |
| P4 그래프 | 그림으로 확인한다 | `plt.hist(...)`, `plt.plot(...)`, `plt.xlabel(...)`, `plt.show()` |
| P5 모델 | 데이터로 학습하고 예측한다 | `model.fit(X, y)`, `model.predict(X)` |

연습용 데이터는 모듈 3에서 쓸 **압출기 필터 고장 실험 60회의 목록**입니다. 한 행이 실험 한 번입니다.

### P1 · 읽고 보기

`pd.read_csv('파일')`은 CSV 파일을 표(DataFrame)로 읽습니다. `.head()`는 앞의 5행, `.shape`는 (행 수, 열 수)입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
runs = pd.read_csv('data/filter_runs.csv')
print(runs.shape)
runs.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — `(60, 5)`: 실험 60회, 열 5개입니다.

| 열 | 뜻 |
|---|---|
| `run_id` | 실험 번호 |
| `failure_mode` | 고장 형태. `clog` = 필터 막힘, `burst` = 필터 파열 |
| `failure_s` | 고장까지 걸린 시간(초). 이 실험의 **수명**입니다 |
| `status`, `split` | 모듈 3에서 쓰는 구분(분석 대상 여부, 학습/시험) |

### P2 · 골라내기

`runs['failure_mode'] == 'clog'` 는 행마다 참(True)/거짓(False)을 돌려줍니다. 이것을 `runs[ ... ]` 안에 넣으면 **참인 행만** 남습니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
clog = runs[runs['failure_mode'] == 'clog']
print('막힘 실험 수:', len(clog))
clog.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 60회 중 42회가 막힘, 나머지 18회가 파열입니다. `len(표)`는 행의 개수입니다.

### P3 · 묶어서 요약

"고장 형태별 평균 수명"처럼 **그룹마다 같은 계산**을 할 때 `groupby`를 씁니다.
읽는 순서: `runs`를 `failure_mode`로 묶어서 → `failure_s` 열의 → 평균.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
runs.groupby('failure_mode')['failure_s'].mean()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 막힘의 평균 수명은 약 547초, 파열은 약 269초입니다. `.mean()` 대신 `.median()`(중앙값), `.count()`(개수), `.min()`, `.max()`도 똑같이 쓸 수 있습니다.

### P4 · 그래프

평균 하나로는 "얼마나 흩어져 있는지"를 알 수 없습니다. **히스토그램**으로 수명의 분포를 봅니다.
그래프 코드는 항상 **그리기 → 축 이름 → 보여 주기** 순서입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
plt.hist(clog['failure_s'], bins=15)
plt.xlabel('수명 (초)')
plt.ylabel('실험 수')
plt.title('필터 막힘 실험의 수명 분포')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 같은 조건으로 시험한 필터인데도 수명이 약 115초부터 1,600초까지 **10배 이상** 차이 납니다.
모든 필터를 "300초마다 교체"하면 어떤 필터는 그 전에 막히고, 어떤 필터는 멀쩡한데 버려집니다.
**그래서 시간 대신 상태를 보고 정비(CBM)** 하려는 것입니다. 이 과정의 출발점입니다.

### P5 · 모델 (미리 보기)

머신러닝 모델은 모두 같은 세 줄로 씁니다. 모듈 2부터 직접 입력합니다.

```python
model = 모델이름(설정)           # 1) 모델 만들기
model.fit(X_train, y_train)      # 2) 학습: 입력 X와 정답 y의 관계를 배운다
pred = model.predict(X_test)     # 3) 예측: 새 입력에 대해 답을 낸다
```

---
## 4. 실습 과제 (10분)

### 📝 과제 1 · 막힘 실험의 수명 요약
막힘 실험(`clog`)의 수명 **최솟값, 중앙값, 최댓값**을 구하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: clog['failure_s'].min() 처럼 씁니다. .median(), .max()도 같은 방식입니다.


### 📝 과제 2 · 파열 실험의 히스토그램
P2와 P4를 이어서, **파열(`burst`) 실험만** 골라 수명 히스토그램을 그리세요. 축 이름과 제목도 붙입니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트 1: burst = runs[runs['failure_mode'] == ___]
# 힌트 2: 위의 P4 코드에서 clog를 burst로 바꿉니다.


### 📝 과제 3 · 두 분포를 겹쳐 그리기 (도전)
막힘과 파열의 히스토그램을 **한 그림에 겹쳐** 그려 보세요. `plt.hist(..., alpha=0.5, label='막힘')`처럼 투명도와 이름표를 주고, 마지막에 `plt.legend()`를 부르면 됩니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: plt.hist를 두 번 부른 뒤 plt.legend(), plt.show()


### 📝 과제 4 · 생각해 보기
"모든 필터를 **200초마다** 교체한다"는 규칙을 쓰면, 막힘 실험 42회 중 몇 회가 교체 전에 고장 날까요? 코드로 세어 보고, 이 규칙의 문제를 한 문장으로 적으세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: (clog['failure_s'] < 200).sum() — 조건이 참인 개수를 셉니다.


✏️ **나의 답:** (여기를 두 번 클릭해서 적으세요)

---
## ✅ 정리

- Colab은 **위에서부터 순서대로**, 초기화되면 **준비 셀부터 다시**.
- 분석 코드는 **P1 읽기 → P2 고르기 → P3 묶어 요약 → P4 그래프 → P5 모델**의 조합입니다.
- 같은 조건의 필터도 수명이 10배 넘게 다릅니다. 고정 주기 교체의 한계가 **상태기반정비(CBM)**가 필요한 이유입니다.

다음: **1-1 · 하드디스크 6만 대의 고장 기록과 고장률**